# UdyamSetu Indic TTS API (Google Colab)

This notebook serves **Hindi (`hi`)**, **Marathi (`mr`)**, and **English (`en`)** speech with AI4Bharat Indic Parler-TTS. It starts an authenticated `POST /tts` API and creates a temporary public HTTPS URL with ngrok.

> Keep this Colab notebook private. A Colab runtime and its public URL stop when the runtime is disconnected or times out. For a production deployment, host the same FastAPI service on a persistent GPU service.

## Before running

1. In Colab, select **Runtime → Change runtime type → T4 GPU**.
2. Create a free ngrok account and copy its authtoken from the ngrok dashboard.
3. Run the cells in order.

In [ ]:
# Install the model runtime and the small HTTP/tunnel server.
!pip -q install git+https://github.com/huggingface/parler-tts.git soundfile fastapi 'uvicorn[standard]' pyngrok nest-asyncio


In [ ]:
import os
import secrets
from getpass import getpass

# Required to publish the temporary HTTPS URL. Do not commit this value.
os.environ['NGROK_AUTHTOKEN'] = getpass('Paste your ngrok authtoken: ')

# This is the API key your UdyamSetu backend will send in X-API-Key.
# Save it now; it is generated fresh each time the runtime starts.
TTS_API_KEY = secrets.token_urlsafe(32)
print('TTS API key (store this securely):', TTS_API_KEY)


In [ ]:
import io
import torch
import soundfile as sf
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer

MODEL_ID = 'ai4b-hf/indic-parler-tts-pretrained-v3'
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cpu':
    print('WARNING: No GPU found. Speech generation will be slow; use a Colab T4 GPU for the API.')

model = ParlerTTSForConditionalGeneration.from_pretrained(MODEL_ID).to(DEVICE).eval()
prompt_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
description_tokenizer = AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path)
SAMPLE_RATE = model.config.sampling_rate
print(f'Model ready on {DEVICE}; sample rate: {SAMPLE_RATE} Hz')


In [ ]:
# A language-specific description helps keep the generated voice natural and consistent.
VOICE_DESCRIPTIONS = {
    'en': 'Mary speaks English in a warm, clear, helpful voice at a moderate pace. The recording is very high quality with no background noise.',
    'hi': 'Divya speaks Hindi in a warm, clear, helpful voice at a moderate pace. The recording is very high quality with no background noise.',
    'mr': 'Sunita speaks Marathi in a warm, clear, helpful voice at a moderate pace. The recording is very high quality with no background noise.',
}

@torch.inference_mode()
def synthesize_wav(text: str, language: str) -> bytes:
    if language not in VOICE_DESCRIPTIONS:
        raise ValueError('language must be one of: en, hi, mr')
    text = text.strip()
    if not text:
        raise ValueError('text cannot be empty')
    if len(text) > 600:
        raise ValueError('text is limited to 600 characters per request')

    description_ids = description_tokenizer(VOICE_DESCRIPTIONS[language], return_tensors='pt').input_ids.to(DEVICE)
    prompt_ids = prompt_tokenizer(text, return_tensors='pt').input_ids.to(DEVICE)
    audio = model.generate(input_ids=description_ids, prompt_input_ids=prompt_ids)
    buffer = io.BytesIO()
    sf.write(buffer, audio.cpu().numpy().squeeze(), SAMPLE_RATE, format='WAV', subtype='PCM_16')
    buffer.seek(0)
    return buffer.read()

# Quick local test before exposing the endpoint.
test_audio = synthesize_wav('नमस्कार! मी उद्यमसेतू सहाय्यक आहे.', 'mr')
from IPython.display import Audio, display
display(Audio(test_audio, rate=SAMPLE_RATE))


In [ ]:
import threading
import nest_asyncio
import uvicorn
from fastapi import FastAPI, Header, HTTPException
from fastapi.responses import Response
from pydantic import BaseModel, Field
from pyngrok import ngrok

app = FastAPI(title='UdyamSetu Indic TTS', version='1.0')

class TTSRequest(BaseModel):
    text: str = Field(min_length=1, max_length=600)
    language: str = Field(pattern='^(en|hi|mr)$')

def authorize(x_api_key: str | None):
    if not x_api_key or not secrets.compare_digest(x_api_key, TTS_API_KEY):
        raise HTTPException(status_code=401, detail='Invalid API key')

@app.get('/health')
def health():
    return {'status': 'ok', 'languages': ['en', 'hi', 'mr']}

@app.post('/tts')
def text_to_speech(payload: TTSRequest, x_api_key: str | None = Header(default=None)):
    authorize(x_api_key)
    try:
        wav = synthesize_wav(payload.text, payload.language)
        return Response(content=wav, media_type='audio/wav', headers={'Cache-Control': 'private, max-age=86400'})
    except ValueError as error:
        raise HTTPException(status_code=400, detail=str(error))

nest_asyncio.apply()
server = uvicorn.Server(uvicorn.Config(app, host='0.0.0.0', port=8001, log_level='warning'))
thread = threading.Thread(target=server.run, daemon=True)
thread.start()

ngrok.set_auth_token(os.environ['NGROK_AUTHTOKEN'])
tunnel = ngrok.connect(8001, bind_tls=True)
PUBLIC_TTS_URL = tunnel.public_url
print('Public TTS endpoint:', f'{PUBLIC_TTS_URL}/tts')
print('Health endpoint:', f'{PUBLIC_TTS_URL}/health')
print('\nAdd these to the repository-root .env file (never commit the API key):')
print(f'INDIC_TTS_URL={PUBLIC_TTS_URL}/tts')
print(f'INDIC_TTS_API_KEY={TTS_API_KEY}')


In [ ]:
# Test the public API exactly as your UdyamSetu backend will call it.
import requests

response = requests.post(
    f'{PUBLIC_TTS_URL}/tts',
    headers={'X-API-Key': TTS_API_KEY},
    json={'text': 'नमस्कार! मी उद्यमसेतू सहाय्यक आहे.', 'language': 'mr'},
    timeout=180,
)
response.raise_for_status()
display(Audio(response.content, rate=SAMPLE_RATE))


## API contract

```http
POST {INDIC_TTS_URL}
X-API-Key: {INDIC_TTS_API_KEY}
Content-Type: application/json

{"text": "आपकी सहायता के लिए मैं यहाँ हूँ।", "language": "hi"}
```

The response body is `audio/wav`. Add the printed values to the repository-root `.env` file (next to `.env.example`). The UdyamSetu backend should proxy that audio to the frontend; do not expose `INDIC_TTS_API_KEY` to the browser.

### Operational notes

- The generated URL is temporary and changes after a Colab restart. Update `INDIC_TTS_URL` whenever it changes.
- Colab is suitable for demos, not production: it can disconnect and has limited concurrency.
- Keep text short (1–3 sentences) for lower latency. The endpoint enforces a 600-character request limit.
- Your chatbot must send Devanagari Hindi/Marathi text for best pronunciation, not Latin transliterations.